#### 1. Bronze processing

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
%run /Workspace/Users/dhotepatil00@gmail.com/regis-healthcare/1_setup/utility

In [0]:
print(bronze_schema,silver_schema,gold_schema) 

In [0]:
dbutils.widgets.text("catalog","regis_healthcare","catalog")
dbutils.widgets.text("data_source","admissions","data_source")

In [0]:
catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")

In [0]:
# Define S3 path
bucket = "regis-healthcare"
prefix = f"source-row-data /{data_source}"
s3_path = f"s3://{bucket}/{prefix}/"

# Use dbutils to list files and find the latest
files = dbutils.fs.ls(s3_path)
latest_file = sorted(files, key=lambda x: x.modificationTime, reverse=True)[0]

base_path = latest_file.path
print("Latest file path:", base_path)

In [0]:
df = (
    spark.read.format("csv")
       .option("header",True)
       .option("inferSchema",True)
       .load(base_path)
       .withColumn("current_date",F.current_date())
       .withColumn("read_timestamp",F.current_timestamp())
       .select("*","_metadata.file_name","_metadata.file_size")
    )
print(df.count())
display(df.limit(10))

In [0]:
df.printSchema()

In [0]:
df.write\
    .format("delta")\
            .mode("overwrite")\
                .saveAsTable(f"{catalog}.{bronze_schema}.{data_source}")
# .option("delta.enableChangeDataFeed","true")\

In [0]:
# bronze write to s3
df.write.format("delta")\
    .option("overwriteSchema","true")\
    .mode("overwrite")\
    .partitionBy("current_date")\
    .save(f"s3://regis-healthcare/bronze-row-data/{data_source}/")

#### 2. Silver Processing

In [0]:
df_bronze = spark.sql(f"select * from {catalog}.{bronze_schema}.{data_source};")
display(df_bronze)
print(df_bronze.count())

In [0]:
# schema check
print(df_bronze.count())
df_bronze.printSchema()

In [0]:
df_bronze.columns

In [0]:
# drop duplicate
df_silver = df_bronze.dropDuplicates()
print(df_silver.count())

In [0]:
df_silver = df_silver.withColumn(
    "admission_id",
    F.trim(F.col("admission_id"))
).withColumn(
    "resident_id",
    F.trim(F.col("resident_id"))
).withColumn(
    "facility_id",
    F.trim(F.col("facility_id"))
).withColumn(
    "admission_date",
    F.trim(F.col("admission_date"))
).withColumn(
    "admission_time",
    F.trim(F.col("admission_time"))
).withColumn(
    "admission_type",
    F.trim(F.col("admission_type"))
).withColumn(
    "referring_doctor",
    F.trim(F.col("referring_doctor"))
).withColumn(
    "ward",
    F.trim(F.col("ward"))
).withColumn(
    "bed_number",
    F.trim(F.col("bed_number"))
).withColumn(
    "care_plan_id",
    F.trim(F.col("care_plan_id"))
).withColumn(
    "assessed_care_level",
    F.trim(F.col("assessed_care_level"))
).withColumn(
    "source_of_referral",
    F.trim(F.col("source_of_referral"))
).withColumn(
    "notes",
    F.trim(F.col("notes"))
).withColumn(
    "created_by",
    F.trim(F.col("created_by"))
).withColumn(
    "created_at",
    F.trim(F.col("created_at"))
)

In [0]:
# null records count 
from pyspark.sql.functions import col,count,when
null_count = df_silver.select([count(when(col(c).isNull(),c)).alias(c)for c in df_silver.columns
                               ])
display(null_count)

#### Cleaning data in table

In [0]:
# admission_id
check = df_silver.filter(col("admission_id").rlike("^\\ADM"))
display(check)
df_silver = df_silver.withColumn("admission_id",when(col("admission_id").rlike("^\\ADM"),None).otherwise(col("admission_id")))
display(df_silver)

In [0]:
# resident_id
check = df_silver.filter(col("resident_id").rlike("^\\RES"))
display(check)
df_silver = df_silver.withColumn("resident_id",when(col("resident_id").rlike("^\\RES"),None).otherwise(col("resident_id")))
display(df_silver)

In [0]:
# facility_id
check = df_silver.filter(~col("facility_id").rlike("^FAC"))
display(check)
df_silver = df_silver.withColumn("facility_id",when(~col("facility_id").rlike("^FAC"),None).otherwise(col("facility_id")))
display(df_silver)

In [0]:
# admission_date
from pyspark.sql.functions import col,when
from pyspark.sql import functions as F
df_silver = df_silver.withColumn(
    "admission_date",
    F.coalesce(
        # Date-only formats
        F.try_to_date(F.trim(F.col("admission_date")), F.lit("yyyy/MM/dd")),
        F.try_to_date(F.trim(F.col("admission_date")), F.lit("dd/MM/yyyy")),
        F.try_to_date(F.trim(F.col("admission_date")), F.lit("yyyy-MM-dd")),
        F.try_to_date(F.trim(F.col("admission_date")), F.lit("dd-MM-yyyy")),
        # Timestamp formats
        F.try_to_timestamp(F.trim(F.col("admission_date")), F.lit("yyyy-MM-dd HH:mm:ss")),
        F.try_to_timestamp(F.trim(F.col("admission_date")), F.lit("yyyy/MM/dd HH:mm:ss"))
    )
)
df_silver = df_silver.withColumn("admission_date", F.to_date("admission_date"))
display(df_silver)

In [0]:
# admission_time
from pyspark.sql import functions as F
# Extract time (HH:mm:ss) from timestamp column
df_silver = df_silver.withColumn("admission_time", F.date_format("admission_time", "HH:mm:ss"))
display(df_silver)

In [0]:
# admission_type
from pyspark.sql.functions import col,when,upper
df_silver = df_silver.withColumn("admission_type",upper(col("admission_type")))
dup = df_silver.groupBy("admission_type").count()
display(dup)

In [0]:
# referring_doctor
check = df_silver.filter(col("resident_id").rlike("^\\Dr"))
display(check)

In [0]:
# ward
dup = df_silver.groupBy("ward").count()
display(dup)

In [0]:
# bed_number
dup = df_silver.groupBy("bed_number").count()
display(dup)

In [0]:
# care_plan_id
check = df_silver.filter(~col("care_plan_id").rlike("^CP"))
display(check)


In [0]:
# assessed_care_level
dup = df_silver.groupBy("assessed_care_level").count()
display(dup)

In [0]:
# source_of_referral
dup = df_silver.groupBy("source_of_referral").count()
display(dup)

In [0]:
# notes
from pyspark.sql.functions import col, when, concat, lit
# Fill notes if null
df_silver = df_silver.withColumn(
    "notes",
    when(
        col("notes").isNull(),
        concat(lit("Notes for admission "), col("admission_id"))
    ).otherwise(col("notes"))
)

dup = df_silver.groupBy("notes").count().filter(col("count")>1)
display(dup)


In [0]:
# created_by
check = df_silver.filter(~col("created_by").rlike("^EMP"))
display(check)

In [0]:
# created_at
from pyspark.sql.functions import col, to_timestamp
df_silver = df_silver.withColumn(
    "created_at",
    to_timestamp(col("created_at"), "yyyy-MM-dd HH:mm:ss")  # specify format if needed
)
display(df_silver)


#### Silver table load

In [0]:
df_silver.write\
    .format("delta")\
        .option("delta.enableChangeDataFeed","true")\
            .option("mergeSchema","true")\
                .option("overwriteSchema","true")\
            .mode("overwrite")\
               .saveAsTable(f"{catalog}.{silver_schema}.{data_source}")

dt = spark.sql(f"select * from {catalog}.{silver_schema}.{data_source};")
print(dt.count())
display(dt)

In [0]:
# load to s3
df_silver.write.format("delta")\
    .option("mergeSchema","true")\
    .option("overwriteSchema","true")\
    .mode("overwrite")\
    .partitionBy("current_date")\
    .save(f"s3://regis-healthcare/silver-clean-data/{data_source}/")

#### Gold Processing

In [0]:
df_silver = spark.sql(f"select * from {catalog}.{silver_schema}.{data_source};")
print(df_silver.count())

In [0]:
df_gold = df_silver.select(
'admission_id',
 'resident_id',
 'facility_id',
 'admission_date',
 'admission_time',
 'admission_type',
 'referring_doctor',
 'ward',
 'bed_number',
 'care_plan_id',
 'assessed_care_level',
 'source_of_referral',
 'notes',
 'created_by',
 'created_at'
)
display(df_gold)

In [0]:
# df_gold.write\
#     .format("delta")\
#     .option("mergeSchema","true")\
#     .option("overwriteSchema","true")\
#         .option("delta.enableChangeDataFeed","true")\
#             .mode("overwrite")\
# .saveAsTable(f"{catalog}.{gold_schema}.fact_{data_source}")

In [0]:
df_gold.write\
    .format("delta")\
    .option("mergeSchema","true")\
    .option("overwriteSchema","true")\
        .option("delta.enableChangeDataFeed","true")\
            .mode("overwrite")\
.saveAsTable(f"{catalog}.{gold_schema}.sb_fact_{data_source}")
print(df_gold.count())

In [0]:
df = spark.sql(f"select * from {catalog}.{gold_schema}.sb_fact_{data_source};")
print(df.count())

In [0]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F

# Load Delta table with correct fully-qualified name
delta_table = DeltaTable.forName(spark, "caroucell_pro.gold.fact_admissions")
# Create DataFrame from source table with correct fully-qualified name
# sb_dim_products
df_child_products = (
    spark.table("caroucell_pro.gold.sb_fact_admissions")
    .select("*")
)
# Perform merge
delta_table.alias("target").merge(
    source=df_child_products.alias("source"),
    condition="target.admission_id = source.admission_id"
).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

In [0]:
dim_df = spark.sql(f"select * from {catalog}.{gold_schema}.fact_{data_source};")
print(dim_df.count())

sb_dim_df = spark.sql(f"select * from {catalog}.{gold_schema}.sb_fact_{data_source};")
print(sb_dim_df.count())

#### gold load to s3

In [0]:
# df_gold.write\
#     .format("delta")\
#     .option("mergeSchema","true")\
#     .option("overwriteSchema","true")\
#         .option("delta.enableChangeDataFeed","true")\
#             .mode("overwrite")\
# .save(f"s3://regis-healthcare/gold-delta-table/fact_{data_source}")

In [0]:
display(df)

In [0]:
from delta.tables import DeltaTable

# ✅ Path to your Delta table stored in S3
delta_table_path = f"s3://regis-healthcare/gold-delta-table/fact_{data_source}"

# ✅ Load target Delta table
delta_table = DeltaTable.forPath(spark, delta_table_path)

# ✅ Source DataFrame (example: df_child_products)
source_df = df

# ✅ Perform MERGE with upsert logic
(
    delta_table.alias("target")
    .merge(
        source_df.alias("source"),
        "target.admission_id = source.admission_id"
    )
    .whenMatchedUpdateAll()      # Update all columns when matched
    .whenNotMatchedInsertAll()   # Insert all columns when not matched
    .execute()
)
